In [1]:
import os
import pandas as pd
from pathlib import Path
from maomao.parsing.parsing_utils import *
from maomao.utils.constants import *

#### Processing and standardizing peptide datasets (iAMPCN)

This notebook curates the **iAMPCN** source by assembling multiple task-specific peptide datasets from heterogeneous inputs. The source provides FASTA files organized by task (hemolytic, cytotoxic, toxic) and by split (train/test). Here we parse and standardize all inputs, harmonize label conventions, perform duplicate consistency checks per task, and export curated datasets and metadata for downstream analysis.

- **Toxic effect / endpoint:** hemolytic, cytotoxic, and toxic
- **Source:** iAMPCN
- **Sequence scope:** only non-modified peptide sequences are retained for the final dataset.

The pipeline performs the following steps:

- **Loads multiple FASTA files** across tasks and splits using a helper loader:
  - assigns `label = 0` for files whose path/name contains `"neg"`,
  - assigns `label = 1` otherwise,
  - stores the originating `source_file` for traceability.
- **Builds task-specific datasets**:
  - `hemolytic`
  - `cytotoxic`
  - `toxic`
- **Checks duplicated sequences independently per task**:
  - unique sequences are retained,
  - duplicates with consistent labels are collapsed,
  - sequences with conflicting labels are flagged as errors.
- **Builds metadata** from the project-wide Excel description sheet and appends QC statistics.
- **Exports curated outputs**:
  - `processed_hemolytic_dataset.csv`
  - `processed_cytotoxic_dataset.csv`
  - `processed_toxic_dataset.csv`
  - `detected_error_sequences.csv`
  - `metadata.json`

In [2]:
name_source = "iAMPCN"
name_task = "toxic_effect_classification"

# PATH_INPUT and PATH_EXPORT are imported from maomao.utils.constants
# Update them in constants.py according to the required input and export paths.

- Reading raw data

In [3]:
def read_multi_files(files):
    dfs = []
    for file in files:
        path = Path(PATH_INPUT) / name_source / file
        df = read_fasta_doc(path)
        df = df.assign(
            label=0 if "neg" in path.name.lower() else 1,
            source_file=path.name
        )
        dfs.append(df)
    df_fasta = pd.concat(dfs, ignore_index=True)
    return df_fasta

In [4]:
files = [
    "AMP_2nd_test/cytotoxic/neg.fasta",
    "AMP_2nd_test/cytotoxic/pos.fasta",
    "AMP_2nd_train/cytotoxic/neg_cdhit_100.fasta",
    "AMP_2nd_train/cytotoxic/pos_cdhit_100.fasta",
]

df_cytotoxic = read_multi_files(files)
df_cytotoxic.shape

(22404, 4)

In [5]:
files = [
    "AMP_2nd_test/hemolytic/neg.fasta",
    "AMP_2nd_test/hemolytic/pos.fasta",
    "AMP_2nd_train/hemolytic/neg_cdhit_100.fasta",
    "AMP_2nd_train/hemolytic/pos_cdhit_100.fasta",
]

df_hemolytic = read_multi_files(files)
df_hemolytic.shape

(22646, 4)

In [6]:
files = [
    "AMP_2nd_train/toxic/neg_cdhit_100.fasta",
    "AMP_2nd_train/toxic/pos_cdhit_100.fasta",
]

df_toxic = read_multi_files(files)
df_toxic.shape

(16542, 4)

- Concatenating dataset

In [7]:
df_iampcn = pd.concat([
    df_cytotoxic,
    df_hemolytic,
    df_toxic
],ignore_index=True)

- Checking duplicates

In [8]:
df_remove_duplicated_toxic, df_errors_toxic, df_unique_toxic = processing_duplicated(df_toxic, group_seq="sequence", sort_key="label")

In [9]:
df_remove_duplicated_cytotoxic, df_errors_cytotoxic, df_unique_cytotoxic = processing_duplicated(df_cytotoxic, group_seq="sequence", sort_key="label")

In [10]:
df_remove_duplicated_hemolytic, df_errors_hemolytic, df_unique_hemolytic = processing_duplicated(df_hemolytic, group_seq="sequence", sort_key="label")

In [11]:
df_full_hemolytic = pd.concat([df_unique_hemolytic, df_remove_duplicated_hemolytic])
df_full_toxic = pd.concat([df_unique_toxic, df_remove_duplicated_toxic])
df_full_cytotoxic = pd.concat([df_unique_cytotoxic, df_remove_duplicated_cytotoxic])
df_full = pd.concat([df_full_hemolytic, df_full_toxic, df_full_cytotoxic])
df_errors = pd.concat([df_errors_hemolytic, df_errors_toxic, df_errors_cytotoxic])

In [12]:
df_errors.shape

(0, 1)

- Working with metada

In [13]:
df_metada = read_metadata("../../raw_data/raw_data_description.xlsx", name_source)
dict_metadata = create_metada_with_multiple_values(df_metada)

In [14]:
raw_total_sequences = (
    len(df_cytotoxic)
    + len(df_hemolytic)
    + len(df_toxic)
)

In [15]:
dict_metadata.update({
    "number_of_raw_sequences": int(raw_total_sequences),
    "number_of_sequences_retained": len(df_full),
    "number_of_positive_sequences": int((df_full["label"] == 1).sum()),
    "number_of_negative_sequences": int((df_full["label"] == 0).sum()),
    "number_of_erroneous_sequences" : len(df_errors),
    "modified_sequences_included": False,
})

dict_metadata

{'type source': 'Dataset',
 'static-dynamic': 'Static',
 'license': 'No information',
 'year of publication': 2023,
 'last update date': datetime.datetime(2023, 3, 2, 0, 0),
 'download date': Timestamp('2025-06-01 00:00:00'),
 'file format': 'csv;fasta',
 'peptide property': 'insecticidal, toxic;cytotoxic, toxic;hemolytic, toxic;toxic',
 'dataset information': 'Positive, Negative;Negative;Positive',
 'unit of measurement': 'No information',
 'obtaining negative dataset': 'Sampling from uniprot;No information',
 'repository or server': 'https://drive.google.com/drive/folders/16GCaw51QJaiN3ypUwlQmehML--w3FBUd',
 'publication': 'https://academic.oup.com/bib/article/24/4/bbad240/7208684',
 'number_of_raw_sequences': 61592,
 'number_of_sequences_retained': 61592,
 'number_of_positive_sequences': 2662,
 'number_of_negative_sequences': 58930,
 'number_of_erroneous_sequences': 0,
 'modified_sequences_included': False}

- Exporting data

In [16]:
os.makedirs(f"{PATH_EXPORT}/{name_task}/{name_source}/", exist_ok=True)
export_json(f"{PATH_EXPORT}/{name_task}/{name_source}/metadata.json", dict_metadata)

In [17]:
df_full_hemolytic.to_csv(f"{PATH_EXPORT}/{name_task}/{name_source}/processed_hemolytic_dataset.csv", index=False)
df_full_toxic.to_csv(f"{PATH_EXPORT}/{name_task}/{name_source}/processed_toxic_dataset.csv", index=False)
df_full_cytotoxic.to_csv(f"{PATH_EXPORT}/{name_task}/{name_source}/processed_cytotoxic_dataset.csv", index=False)

df_errors.to_csv(f"{PATH_EXPORT}/{name_task}/{name_source}/detected_error_sequences.csv", index=False)